# 1.项目背景与目标 (Markdown)
S2.4 深度分析：A 类风险场景的分组解读与可视化
## 1. 目标 
基于 S2.3 筛选出的 A 类高优先级风险场景（Class A Risk Scenarios），进行类似参考论文（Table 7-11）的深度解读：

- 宏观分布：统计各类事故的风险场景数量，确定治理重点。

- 微观画像：针对核心事故类型（如脱轨、平交道口事故），绘制专属的风险柱状图。

- 致因叙事：自动生成具体的风险致因描述文本，用于报告撰写。

## 2. 输入数据

s2_3_final_pareto_risk_scenarios.csv (S2.3 的产出文件)

# 2. 加载数据与环境设置 (Code)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast

# --- 绘图设置 ---
sns.set(style="whitegrid")
# 解决中文显示问题 (根据您的系统选择字体，Windows通常是SimHei，Mac是Arial Unicode MS)
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False

# --- 1. 定义路径 ---
# 根据您之前的目录结构
INPUT_FILE = os.path.join('..', 'results', 'reports', 's2_3_final_pareto_risk_scenarios.csv')
OUTPUT_FIG_DIR = os.path.join('..', 'results', 'reports', 'figures')

# 确保图片输出目录存在
os.makedirs(OUTPUT_FIG_DIR, exist_ok=True)

# --- 2. 加载数据 ---
try:
    df_risks = pd.read_csv(INPUT_FILE)
    print(f"成功加载 A 类风险场景数据，共 {len(df_risks)} 条记录。")
    
    # 简单预览
    # display(df_risks.head())
    
except FileNotFoundError:
    print(f"错误：找不到文件 {INPUT_FILE}。请先运行 S2.3 笔记本生成该文件。")

# 3. 宏观分析：风险分布概览 (Code)
分析逻辑：我们要回答“哪种事故类型的 A 类风险最多？”。这对应论文中的 Figure 5 (Number of rules by cause)。

In [ ]:
# --- 统计每种事故类型的 A 类场景数量 ---
type_counts = df_risks['accident_type'].value_counts().reset_index()
type_counts.columns = ['Accident Type', 'Count']

# --- 绘制横向柱状图 ---
plt.figure(figsize=(10, 6))
sns.barplot(x='Count', y='Accident Type', data=type_counts, palette='viridis')

plt.title('Distribution of Class A Risk Scenarios by Accident Type', fontsize=14)
plt.xlabel('Number of High-Priority Scenarios', fontsize=12)
plt.ylabel('Accident Type', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)

# 保存并显示
save_path = os.path.join(OUTPUT_FIG_DIR, 'S2.4_Risk_Distribution.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

print("解读：柱子最长的类别（通常是 Derailment），就是我们需要在报告中花费最多篇幅去分析的重点领域。")

# 4. 微观分析：核心场景的分组可视化 (Code)
分析逻辑：这是论文最核心的可视化部分。我们需要把混在一起的数据拆开，针对前 3-4 个主要事故类型，分别画出它们最致命的 Top 5 风险组合。

In [ ]:
# --- 定义一个可视化函数 ---
def plot_top_risks_for_type(df, accident_type, top_n=5):
    """
    针对特定的事故类型，绘制 Top N 风险场景的 CRI 得分图。
    """
    # 1. 筛选并排序
    df_subset = df[df['accident_type'] == accident_type].copy()
    df_subset = df_subset.sort_values(by='CRI_Score', ascending=False).head(top_n)
    
    if df_subset.empty:
        print(f"提示：事故类型 '{accident_type}' 没有数据。")
        return

    # 2. 处理 Evidence 文本，让图表标签更易读
    # 把 "{'Key': 'Val'}" 简化为 "Key: Val" 并换行
    def format_label(s):
        try:
            d = ast.literal_eval(s)
            # 将字典转换为多行字符串
            return "\n".join([f"{k}: {v}" for k, v in d.items()])
        except:
            return s[:20] + "..."

    df_subset['label'] = df_subset['evidence'].apply(format_label)

    # 3. 绘图
    plt.figure(figsize=(8, 0.8 * top_n))  # 根据条目数量自动调整高度
    # 使用 CRI Score 作为长度，颜色映射 Uplift (提升度)
    bars = plt.barh(df_subset['label'], df_subset['CRI_Score'], color='cornflowerblue', alpha=0.8)
    
    plt.title(f"Top {top_n} Risk Scenarios: {accident_type}", fontsize=14)
    plt.xlabel("Composite Risk Index (CRI)", fontsize=12)
    plt.gca().invert_yaxis()  # 让第一名排在最上面
    
    # 在柱子上标注数值
    for bar, uplift in zip(bars, df_subset['uplift']):
        plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                 f"Uplift: {uplift:.2f}", va='center', fontsize=10, color='darkred')

    # 保存
    safe_name = accident_type.replace('/', '_').replace(' ', '_')
    save_path = os.path.join(OUTPUT_FIG_DIR, f'S2.4_TopRisks_{safe_name}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# --- 循环绘制主要事故类型 ---
# 只选取数量最多的前 4 类事故进行详细绘图（避免图太多）
top_accident_types = type_counts['Accident Type'].head(4).tolist()

for acc_type in top_accident_types:
    plot_top_risks_for_type(df_risks, acc_type, top_n=5)

# 5. 致因叙事：自动生成解读报告 (Code)
分析逻辑：这是为了帮您写论文/报告的。代码会将数据“翻译”成通顺的句子，您可以直接复制到 Word 里，稍作润色即可。这对应论文中的 "Detailed Interpretation" (Table 7-11)。

In [ ]:
def generate_narrative_report(df, accident_type, top_n=3):
    """
    自动生成文字解读报告
    """
    df_subset = df[df['accident_type'] == accident_type].sort_values('CRI_Score', ascending=False).head(top_n)
    
    print(f"\n### 深度解读：{accident_type} (Top {top_n} Scenarios)")
    print("-" * 60)
    
    for i, (idx, row) in enumerate(df_subset.iterrows(), 1):
        try:
            evidence_dict = ast.literal_eval(row['evidence'])
            conditions = " 且 ".join([f"【{k} 为 {v}】" for k, v in evidence_dict.items()])
        except:
            conditions = row['evidence']
            
        print(f"**场景 {i}**：")
        print(f"- **风险模式**：当 {conditions} 时。")
        print(f"- **数据支撑**：该组合使事故概率提升了 {(row['uplift']*100):.1f}% (Uplift)。")
        print(f"- **核心致因**：模型识别出的高贡献特征包括：")
        
        # 解析 matched_details 来展示具体特征得分
        details = row['matched_details'].split('; ')
        for d in details:
            # 只显示分数较高的特征
            if 'Score' in d:
                print(f"  * {d}")
        
        # 简单的整改建议逻辑 (示例)
        if 'Track' in str(evidence_dict):
            print(f"- **建议措施**：加强对相关轨道区段（如 {evidence_dict.get('Track Type', '该区域')}）的几何尺寸与磨损检查。")
        elif 'Equipment' in str(evidence_dict):
            print(f"- **建议措施**：针对 {evidence_dict.get('Equipment Type', '车辆')} 实施更严格的作业前检查或防溜措施。")
        
        print("")

# --- 生成报告文本 ---
print("============ S2.4 风险致因深度解读报告 ============")
for acc_type in top_accident_types:
    generate_narrative_report(df_risks, acc_type, top_n=3)
print("===================================================")